# Demonstrasi dan bukti 16 langkah N-gram
Pendamping [notebook utama](core/ngram_lm.ipynb) untuk mahasiswa yang sudah memahami Counter dan probabilitas bersyarat. Tujuannya menjelaskan `<UNK>`, menelusuri rumus per token, membandingkan MLE–Laplace, dan memetakan bukti tugas.

Urutan: reproduksi partisi → contoh UNK → perhitungan dan pemeriksaan manual → prediksi → generasi → bukti 16 langkah. Jalankan Run All dari kernel bersih, memakai environment proyek.

Seluruh contoh Brown memakai preprocessing, split 80/20 seed 42, vocabulary train, dan ambang 2 yang sama dengan hasil awal. Model tetap unigram/bigram/trigram, MLE dan Laplace. Hasil tambahan masuk `hasil_demonstrasi/`; notebook utama dan hasil eksperimen lama diperiksa dengan SHA-256 sebelum/sesudah, tanpa ditulis ulang. Evaluasi ulang di sini hanya pemeriksaan model tetap, tanpa tuning atau memilih model berdasarkan test.

In [1]:
import hashlib
import json
import math
import platform
import sys
from collections import Counter
from importlib.metadata import version
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "core").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display

from core.src.data import DATA_DIR, load_corpus, set_seed
from core.src.evaluate import perplexity, sentence_log_probability, sentence_probability
from core.src.generate import generate
from core.src.models import NGramLM
from core.src.ngram import add_boundaries, events, ngrams
from core.src.preprocess import (
    fit_vocabulary,
    oov_rate,
    preprocess,
    replace_unknown,
    split_sentences,
)

set_seed(42)
OUT = ROOT / "hasil_demonstrasi"
OUT.mkdir(exist_ok=True)
baseline = json.loads((ROOT / "core/results/metrics.json").read_text(encoding="utf-8"))
frozen = json.loads((ROOT / "verifikasi/core_manifest.json").read_text(encoding="utf-8"))
protected = json.loads((ROOT / "verifikasi/perapihan.json").read_text(encoding="utf-8"))[
    "preserved_results"
]
for item in protected:
    frozen[item["current"]] = item["sha256"]
for relative, digest in frozen.items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == digest, relative
print(
    "Python",
    platform.python_version(),
    {name: version(name) for name in ("nltk", "pandas", "numpy")},
)
print("File baseline diperiksa:", len(frozen))

Python 3.11.15 {'nltk': '3.10.3', 'pandas': '3.0.6', 'numpy': '2.4.6'}
File baseline diperiksa: 166


## Reproduksi data yang sama — langkah 1–8
Statistik seluruh corpus hanya eksplorasi. Vocabulary, count dan seluruh model dibangun dari train. Pemetaan indeks harus persis sama dengan `core/results/split.json`.

In [2]:
raw = load_corpus("brown")
raw_counts = Counter(word for sentence in raw for word in sentence)
processed, dropped = preprocess(raw)
train, test, train_ids, test_ids = split_sentences(processed, seed=42)
saved_split = json.loads((ROOT / "core/results/split.json").read_text(encoding="utf-8"))
assert {"train_ids": train_ids, "test_ids": test_ids} == saved_split
assert set(train_ids).isdisjoint(test_ids)
assert set(train_ids) | set(test_ids) == set(range(len(processed)))
vocabulary, word_counts = fit_vocabulary(train, min_count=2)
train_mapped = replace_unknown(train, vocabulary)
test_mapped = replace_unknown(test, vocabulary)
models = {
    (n, name): NGramLM(n, alpha=alpha, min_count=1).fit(train_mapped, vocabulary)
    for n in (2, 3)
    for name, alpha in (("MLE", 0), ("Laplace", 1))
}
assert len(vocabulary) == baseline["oov"]["V_targets"]
assert (
    sum(word == "<UNK>" for sentence in train_mapped for word in sentence)
    == baseline["oov"]["train_UNK_tokens"]
)
assert oov_rate(test_mapped, vocabulary) == 0
print(
    "Raw:",
    len(raw),
    "Processed:",
    len(processed),
    "Train/test:",
    len(train),
    len(test),
    "V:",
    len(vocabulary),
)

Raw: 57340 Processed: 56766 Train/test: 45412 11354 V: 22356


## Contoh `<UNK>` nyata — langkah 6
Contoh pertama adalah singleton train; contoh kedua adalah kata test yang sama sekali tidak ada dalam raw vocabulary train. Keduanya menjadi `<UNK>`, dengan probabilitas UNK yang dipelajari dari train. Nol OOV setelah pemetaan berarti semua token terwakili, bukan semua kata asli dikenali.

In [3]:
unknown_rows = []
for name, sentences, ids, predicate in (
    ("singleton train", train, train_ids, lambda word: word_counts[word] == 1),
    ("OOV test", test, test_ids, lambda word: word not in word_counts),
):
    index, position = next(
        (i, j)
        for i, sentence in enumerate(sentences)
        for j, word in enumerate(sentence)
        if predicate(word)
    )
    snippet = sentences[index][max(0, position - 3) : position + 4]
    unknown_rows.append(
        {
            "jenis": name,
            "indeks_kalimat": ids[index],
            "kata_asli": sentences[index][position],
            "count_train": word_counts[sentences[index][position]],
            "sebelum": " ".join(snippet),
            "sesudah": " ".join(replace_unknown([snippet], vocabulary)[0]),
        }
    )
unknown_table = pd.DataFrame(unknown_rows)
display(unknown_table)
unknown_table.to_csv(OUT / "contoh_unk.csv", index=False)

,jenis,indeks_kalimat,kata_asli,count_train,sebelum,sesudah
0,singleton train,4262,meddle,1,every opportunity to meddle in the congo,every opportunity to <UNK> in the congo
1,OOV test,11503,navigable,0,a body of navigable water boats were,a body of <UNK> water boats were


## Rumus yang dapat ditelusuri — langkah 9, 11–15
Setiap baris adalah satu event prediksi, termasuk EOS dan tanpa BOS sebagai target. `count` adalah C(h,w); `context_count` adalah C(h). Numerator/denominator dihitung langsung dari count lalu dicocokkan dengan `model.score()`. Log natural: log(0)=−inf. Produk nol karena event tak terlihat dibedakan dari underflow produk positif.

In [4]:
def probability_trace(model: NGramLM, sentence: list[str]) -> pd.DataFrame:
    rows = []
    for step, gram in enumerate(events(model.map_sentence(sentence), model.n), 1):
        count = model.counts[gram]
        context_count = model.context_counts[gram[:-1]]
        numerator = count + model.alpha
        denominator = context_count + model.alpha * len(model.vocabulary)
        probability = numerator / denominator if denominator else 0.0
        assert math.isclose(probability, model.score(gram[-1], gram[:-1]), abs_tol=1e-12)
        rows.append(
            {
                "step": step,
                "context": " ".join(gram[:-1]),
                "target": gram[-1],
                "count": count,
                "context_count": context_count,
                "numerator": numerator,
                "denominator": denominator,
                "P": probability,
                "log_P": math.log(probability) if probability else -math.inf,
            }
        )
    return pd.DataFrame(rows)


def generation_trace(model: NGramLM, seed: int, max_length: int = 40):
    words = generate(model, seed=seed, max_length=max_length)
    context = model.context(words)
    mass = sum(model.followers.get(context, {}).values()) + model.alpha * len(model.vocabulary)
    reason = (
        "batas panjang"
        if len(words) == max_length
        else "EOS"
        if mass
        else "konteks tanpa probabilitas"
    )
    trace = probability_trace(model, words)
    if reason != "EOS":
        trace = trace.iloc[:-1].copy()  # EOS tidak disampling saat cap/no-mass.
    assert len(trace) == len(words) + (reason == "EOS")
    assert all(trace["P"] > 0)
    assert "<s>" not in words
    return words, reason, trace

### Latihan dengan jawaban terverifikasi
Corpus mini: `a b` dan `a c`; V=5 (`a`, `b`, `c`, UNK, EOS). Hitung P(`a b`) sendiri: MLE = 1 × 1/2 × 1 = 1/2; Laplace = 3/7 × 2/7 × 2/6 = 2/49. Sel berikut memeriksa jawaban, normalisasi seluruh V, serta penghentian generasi. Corpus mini ini khusus sanity check, bukan sumber angka Brown.

In [5]:
mini_mle = NGramLM(2, min_count=1).fit([["a", "b"], ["a", "c"]])
mini_laplace = NGramLM(2, alpha=1, min_count=1).fit([["a", "b"], ["a", "c"]])
assert len(mini_laplace.vocabulary) == 5
for model, expected in ((mini_mle, [1, 0.5, 1]), (mini_laplace, [3 / 7, 2 / 7, 2 / 6])):
    trace = probability_trace(model, ["a", "b"])
    assert all(
        math.isclose(actual, manual) for actual, manual in zip(trace["P"], expected, strict=True)
    )
assert math.isclose(math.prod(probability_trace(mini_laplace, ["a", "b"])["P"]), 2 / 49)
assert probability_trace(mini_mle, ["b", "a"])["P"].min() == 0
for context in [("<s>",), ("a",), ("never",)]:
    assert math.isclose(
        sum(mini_laplace.score(word, context) for word in mini_laplace.vocabulary), 1, abs_tol=1e-9
    )
assert generation_trace(mini_mle, 42)[1] == "EOS"
assert generation_trace(mini_mle, 42, max_length=1)[1] == "batas panjang"
mini_mle.counts.clear()
mini_mle.context_counts.clear()
mini_mle.followers.clear()
assert generation_trace(mini_mle, 42)[1] == "konteks tanpa probabilitas"
print("Sanity check hitungan manual, normalisasi, dan tiga alasan berhenti: lulus.")
manual_checks_passed = True

Sanity check hitungan manual, normalisasi, dan tiga alasan berhenti: lulus.


In [6]:
unseen_sentence, unseen_gram = next(
    (sentence, gram)
    for sentence in test_mapped
    for gram in events(sentence, 2)
    if models[2, "MLE"].counts[gram] == 0
)
trace_frames, sentence_rows = [], []
for n in (2, 3):
    for name in ("MLE", "Laplace"):
        model = models[n, name]
        for case, sentence in (("train_seen", train_mapped[0]), ("test_unseen", unseen_sentence)):
            trace = probability_trace(model, sentence)
            product, log_p = math.prod(trace["P"]), math.fsum(trace["log_P"])
            assert math.isclose(product, sentence_probability(model, sentence), rel_tol=1e-12)
            assert math.isclose(log_p, sentence_log_probability(model, sentence), rel_tol=1e-12)
            manual_pp = math.exp(-log_p / len(trace)) if math.isfinite(log_p) else math.inf
            assert math.isclose(
                manual_pp, perplexity(model, [sentence])["perplexity"], rel_tol=1e-12
            )
            sentence_rows.append(
                {
                    "n": n,
                    "model": name,
                    "case": case,
                    "N": len(trace),
                    "P_sentence": product,
                    "log_P_sentence": log_p,
                    "PP_sentence": manual_pp,
                }
            )
            trace_frames.append(trace.assign(n=n, model=name, case=case))
trace_table = pd.concat(trace_frames, ignore_index=True)
sentence_table = pd.DataFrame(sentence_rows)
print("Kalimat test:", " ".join(unseen_sentence))
display(
    trace_table[(trace_table.n == 2) & (trace_table.case == "test_unseen")].drop(
        columns=["n", "case"]
    )
)
display(sentence_table)
trace_table.to_csv(OUT / "perhitungan_token.csv", index=False)
sentence_table.to_csv(OUT / "perhitungan_kalimat.csv", index=False)

Kalimat test: at one time to most americans unless they were fortunate enough to live near a body of <UNK> water boats were considered the sole concern of fishermen rich people and the united states navy


,step,context,target,count,context_count,numerator,denominator,P,log_P,model
13,1,<s>,at,336,45412,336.0,45412.0,0.007399,-4.906421,MLE
14,2,at,one,36,4235,36.0,4235.0,0.008501,-4.767620,MLE
15,3,one,time,18,2643,18.0,2643.0,0.006810,-4.989298,MLE
16,4,time,to,120,1300,120.0,1300.0,0.092308,-2.382628,MLE
17,5,to,most,9,20879,9.0,20879.0,0.000431,-7.749275,MLE
...,...,...,...,...,...,...,...,...,...,...
91,31,and,the,1850,23041,1851.0,45397.0,0.040774,-3.199720,Laplace
92,32,the,united,308,55780,309.0,78136.0,0.003955,-5.532865,Laplace
93,33,united,states,310,387,311.0,22743.0,0.013675,-4.292220,Laplace
94,34,states,navy,1,475,2.0,22831.0,0.000088,-9.342727,Laplace


,n,model,case,N,P_sentence,log_P_sentence,PP_sentence
0,2,MLE,train_seen,13,9.775559e-24,-52.982157,5.888289e+01
1,2,MLE,test_unseen,35,0.000000e+00,-inf,inf
2,2,Laplace,train_seen,13,1.186056e-44,-101.143110,2.392847e+03
3,2,Laplace,test_unseen,35,2.592180e-118,-270.752542,2.288809e+03
4,3,MLE,train_seen,13,4.782045e-13,-28.368738,8.865884e+00
5,3,MLE,test_unseen,35,0.000000e+00,-inf,inf
6,3,Laplace,train_seen,13,1.547461e-50,-114.692639,6.785279e+03
7,3,Laplace,test_unseen,35,6.650515e-143,-327.374974,1.153996e+04


### Underflow dan perplexity pada partisi tetap — langkah 12–15
Kalimat panjang memakai Laplace agar setiap faktor positif. Perplexity kalimat contoh menjelaskan rumus dan tidak menggantikan evaluasi seluruh test. Nilai Brown di bawah memakai model/partisi tetap yang sama; unigram diperiksa dari log likelihood tersimpan, sedangkan bigram/trigram Laplace dihitung ulang. Normalisasi Laplace memakai seluruh konteks train dengan pengelompokan massa kata tak terlihat yang ekuivalen dengan penjumlahan seluruh vocabulary.

In [7]:
long_sentence = train_mapped[0] * 100
long_trace = probability_trace(models[2, "Laplace"], long_sentence)
assert all(long_trace["P"] > 0)
assert math.prod(long_trace["P"]) == 0
assert math.isclose(
    math.fsum(long_trace["log_P"]), baseline["underflow"]["log_probability"], rel_tol=1e-12
)
long_trace.to_csv(OUT / "underflow_token.csv", index=False)
print("Underflow:", len(long_sentence), "kata; produk = 0; log P =", math.fsum(long_trace["log_P"]))
for n in (2, 3):
    models[n, "Laplace"].assert_laplace_normalized(1e-9)
pp_rows = []
for saved in baseline["perplexity"]:
    label = saved["model"]
    if label == "Unigram Laplace":
        value = math.exp(-saved["log_likelihood"] / saved["predicted_tokens"])
    else:
        n = 3 if label.startswith("Trigram") else 2
        name = "MLE" if label.endswith("MLE") else "Laplace"
        evaluated = perplexity(models[n, name], test_mapped)
        value = evaluated["perplexity"]
        assert evaluated["predicted_tokens"] == saved["predicted_tokens"]
    assert math.isclose(value, float(saved["perplexity"]), rel_tol=1e-12)
    pp_rows.append(
        {
            "model": label,
            "PP": value,
            "N": saved["predicted_tokens"],
            "bukti": "rumus dari hasil beku"
            if label == "Unigram Laplace"
            else "evaluasi ulang model tetap",
        }
    )
pp_table = pd.DataFrame(pp_rows)
display(pp_table)
pp_table.to_csv(OUT / "perplexity_verifikasi.csv", index=False)

Underflow: 1200 kata; produk = 0; log P = -9826.710276870215


,model,PP,N,bukti
0,Bigram MLE,inf,208096,evaluasi ulang model tetap
1,Unigram Laplace,8.458457e+02,208096,rumus dari hasil beku
2,Bigram Laplace,2.633393e+03,208096,evaluasi ulang model tetap
3,Trigram Laplace,1.097793e+04,208096,evaluasi ulang model tetap


## Prediksi MLE–Laplace — langkah 10, 13–14
Kata OOV dipetakan ke UNK: konteks dengan kata asing tidak otomatis menjadi konteks tanpa count. Untuk bigram, konteks EOS dipakai sebagai kasus struktural tanpa event train; untuk trigram, dipakai pasangan kata test yang tidak terlihat di train. Lima skor nol MLE bukan lima prediksi yang memiliki probabilitas positif. Pada konteks tanpa count, Laplace uniform atas V, sehingga urutan alfabetik hanya tie-break.

In [8]:
prediction_rows = []
oov_word = unknown_rows[1]["kata_asli"]
for n in (2, 3):
    cases = (
        ("dikenal", ("the",) if n == 2 else ("in", "the")),
        ("tanpa count konteks", ("</s>",) if n == 2 else tuple(unseen_gram)),
        ("kata OOV", (oov_word,) if n == 2 else ("the", oov_word)),
    )
    for case, context in cases:
        for name in ("MLE", "Laplace"):
            model = models[n, name]
            for rank, (word, probability) in enumerate(model.predict_next(context, 5), 1):
                assert word != "<s>"
                prediction_rows.append(
                    {
                        "n": n,
                        "model": name,
                        "case": case,
                        "context_asli": " ".join(context),
                        "context_model": " ".join(model.context(context)),
                        "context_count": model.context_counts[model.context(context)],
                        "rank": rank,
                        "word": word,
                        "P": probability,
                        "status": "positif" if probability else "nol; tidak ada dukungan MLE",
                    }
                )
prediction_table = pd.DataFrame(prediction_rows)
display(prediction_table)
prediction_table.to_csv(OUT / "prediksi_perbandingan.csv", index=False)

,n,model,case,context_asli,context_model,context_count,rank,word,P,status
0,2,MLE,dikenal,the,the,55780,1,<UNK>,0.030531,positif
1,2,MLE,dikenal,the,the,55780,2,first,0.009591,positif
2,2,MLE,dikenal,the,the,55780,3,same,0.009251,positif
3,2,MLE,dikenal,the,the,55780,4,other,0.006113,positif
4,2,MLE,dikenal,the,the,55780,5,most,0.006024,positif
5,2,Laplace,dikenal,the,the,55780,1,<UNK>,0.021808,positif
6,2,Laplace,dikenal,the,the,55780,2,first,0.006860,positif
7,2,Laplace,dikenal,the,the,55780,3,same,0.006617,positif
8,2,Laplace,dikenal,the,the,55780,4,other,0.004377,positif
9,2,Laplace,dikenal,the,the,55780,5,most,0.004313,positif


## Generasi MLE–Laplace — langkah 16
Lima seed yang sama (42–46) untuk setiap model, cap 40 kata. Sampling tetap memakai fungsi `generate()` asli; jejak menelusuri probabilitas token hasilnya. EOS hanya dicatat bila benar-benar menjadi alasan berhenti, bukan ditambahkan pada sampel yang mencapai cap. Kalimat MLE harus persis sama dengan keluaran awal.

Laplace dapat menghasilkan banyak transisi tidak terlihat karena massa tersebar ke seluruh V. Perbandingan ini mendemonstrasikan perilaku smoothing; koherensi semantik/global belum dinilai oleh manusia.

In [9]:
generation_rows, generation_frames = [], []
for n in (2, 3):
    for name in ("MLE", "Laplace"):
        for i, seed in enumerate(range(42, 47)):
            words, reason, trace = generation_trace(models[n, name], seed)
            if name == "MLE":
                assert words == baseline["generation"][str(n)][i]
            generation_rows.append(
                {
                    "n": n,
                    "model": name,
                    "seed": seed,
                    "kata": len(words),
                    "berhenti": reason,
                    "text": " ".join(words),
                }
            )
            generation_frames.append(trace.assign(n=n, model=name, seed=seed))
            print(f"n={n} {name}, seed={seed}, {len(words)} kata, berhenti={reason}")
            print(" ".join(words), "\n")
generation_table = pd.DataFrame(generation_rows)
generation_table.to_csv(OUT / "generasi_perbandingan.csv", index=False)
pd.concat(generation_frames, ignore_index=True).to_csv(OUT / "generasi_token.csv", index=False)
display(
    generation_table.groupby(["n", "model", "berhenti"], as_index=False).agg(
        jumlah=("seed", "size")
    )
)

n=2 MLE, seed=42, 29 kata, berhenti=EOS
it was before the <UNK> level in both miss schwarzkopf and <UNK> <UNK> and left foot traveler and repair facilities for a joint committee of the stint of christendom 

n=2 MLE, seed=43, 36 kata, berhenti=EOS
somewhere behind that we seldom moved a blind alley for an extraordinary effort to behave in that point charles <UNK> rapidly from squeezing a big one another car with dependable universe is not very troubled time 

n=2 MLE, seed=44, 13 kata, berhenti=EOS
a sudden wave of men like a child continues to the united in 

n=2 MLE, seed=45, 9 kata, berhenti=EOS
for him by the contemporary scene but that curious 

n=2 MLE, seed=46, 4 kata, berhenti=EOS
although bob fogg made 

n=2 Laplace, seed=42, 40 kata, berhenti=batas panjang
it eyebrow choked sharks ordering cavity repelled smiled vacuum ooze study acrylic definitive knoll exacts keelson mending languages avenue seal megaton hoping thwarted loyalties boots fathers bulletins chickasaws plate mad

,n,model,berhenti,jumlah
0,2,Laplace,batas panjang,5
1,2,MLE,EOS,5
2,3,Laplace,batas panjang,5
3,3,MLE,EOS,4
4,3,MLE,batas panjang,1


## Bukti 16 langkah
Status di bawah dibangun dari pemeriksaan yang benar-benar dijalankan dan output notebook utama yang hash-nya tetap identik. Pemeriksaan struktur/output tidak membuktikan kualitas semantik, generalisasi lintas dokumen, atau keunggulan model.

In [10]:
main = json.loads((ROOT / "core/ngram_lm.ipynb").read_text(encoding="utf-8"))
main_codes = [cell for cell in main["cells"] if cell["cell_type"] == "code"]
assert len(main_codes) == 18 and all(cell["execution_count"] is not None for cell in main_codes)
assert not any(
    out["output_type"] == "error" for cell in main_codes for out in cell.get("outputs", [])
)
checks = {}
checks[1] = (
    hashlib.sha256((DATA_DIR / "corpora/brown.zip").read_bytes()).hexdigest()
    == baseline["corpus_sha256"]
    and len(raw) == baseline["raw"]["sentences"]
)
checks[2] = (
    sum(raw_counts.values()) == baseline["raw"]["tokens"]
    and len(raw_counts) == baseline["raw"]["vocabulary"]
)
checks[3] = (
    len(processed) == baseline["processed"]["sentences"]
    and dropped == baseline["processed"]["empty_dropped"]
    and all(word.isalpha() and word == word.lower() for sentence in processed for word in sentence)
)
checks[4] = {"train_ids": train_ids, "test_ids": test_ids} == saved_split
checks[5] = add_boundaries(["the"], 3) == ["<s>", "<s>", "the", "</s>"]
checks[6] = (
    all(word not in vocabulary for word in unknown_table.kata_asli)
    and oov_rate(test_mapped, vocabulary) == 0
    and "<s>" not in vocabulary
)
checks[7] = list(ngrams(["the", "cat", "sleeps"], 2)) == [("the", "cat"), ("cat", "sleeps")]
checks[8] = all(
    sum(models[n, "MLE"].counts.values()) == sum(len(sentence) + 1 for sentence in train)
    for n in (2, 3)
)
checks[9] = manual_checks_passed
checks[10] = len(prediction_table) == 60 and "<s>" not in set(prediction_table.word)
checks[11] = sentence_probability(models[2, "MLE"], unseen_sentence) == 0
checks[12] = math.prod(long_trace["P"]) == 0 and math.isfinite(math.fsum(long_trace["log_P"]))
test_bigrams = Counter(gram for sentence in test_mapped for gram in events(sentence, 2))
unseen_rate = sum(
    count for gram, count in test_bigrams.items() if models[2, "MLE"].counts[gram] == 0
) / sum(test_bigrams.values())
checks[13] = math.isclose(unseen_rate, baseline["unseen_bigrams"]["occurrence_rate"], rel_tol=1e-12)
checks[14] = all(
    model.alpha == 1 and model.score(unseen_gram[-1], unseen_gram[:-1]) > 0
    for (n, name), model in models.items()
    if n == 2 and name == "Laplace"
)
checks[15] = len(pp_table) == 4 and set(pp_table.N) == {
    baseline["perplexity"][0]["predicted_tokens"]
}
checks[16] = len(generation_table) == 20 and set(generation_table.kata) <= set(range(41))
assert all(checks.values()), checks
for relative, digest in frozen.items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == digest, relative
names = [
    "Load corpus",
    "Explore corpus",
    "Preprocessing",
    "Split",
    "Boundary",
    "Unknown words",
    "Generate n-grams",
    "Count n-grams",
    "MLE",
    "Next-word prediction",
    "Sentence probability",
    "Log probability",
    "Zero frequency",
    "Laplace",
    "Perplexity",
    "Text generation",
]
outputs = [
    "Corpus dan hash",
    "Statistik/top-20/histogram/Zipf utama",
    "Contoh preprocessing utama",
    "Indeks split identik",
    "Padding model dan corpus mini",
    "contoh_unk.csv",
    "Contoh n-gram utama dan assert",
    "Counter event train",
    "perhitungan_token.csv",
    "prediksi_perbandingan.csv",
    "perhitungan_kalimat.csv",
    "underflow_token.csv",
    "Unseen rate dan count nol",
    "Normalisasi penuh dan numerator/denominator",
    "perplexity_verifikasi.csv",
    "generasi_perbandingan.csv dan generasi_token.csv",
]
rows = [
    {"langkah": i, "topik": name, "status": "LULUS", "bukti": evidence, "sel_kode_utama": i + 1}
    for i, (name, evidence) in enumerate(zip(names, outputs, strict=True), 1)
]
compliance = pd.DataFrame(rows)
display(compliance)
compliance.to_csv(OUT / "bukti_16_langkah.csv", index=False)
lines = [
    "# Bukti 16 langkah N-gram",
    "",
    "Seluruh pemeriksaan dihasilkan oleh Run All demonstrasi_ngram.ipynb. Nomor sel kode utama adalah urutan sel kode, dimulai dari 1; sel kode 1 adalah bootstrap.",
    "",
    "| Langkah | Topik | Status | Bukti |",
    "|---|---|---|---|",
]
lines += [
    f"| {row['langkah']} | {row['topik']} | {row['status']} | {row['bukti']} |" for row in rows
]
lines += [
    "",
    "Semua file core dan 157 hasil lama tetap SHA-256-identik. Partisi/vocabulary tidak diubah. Koherensi semantik, split dokumen dan generalisasi lintas corpus tidak diuji.",
]
(OUT / "BUKTI_16_LANGKAH.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
(OUT / "verifikasi.json").write_text(
    json.dumps(
        {
            "langkah_lulus": len(checks),
            "sanity_check_manual": manual_checks_passed,
            "hasil_lama_identik": len(protected),
            "core_identik": True,
            "generasi": len(generation_table),
            "prediksi": len(prediction_table),
            "scope": "16 langkah; n=1..3, MLE dan Laplace; tanpa tuning",
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
print("16 langkah: lulus; core dan", len(protected), "hasil lama tetap identik.")

,langkah,topik,status,bukti,sel_kode_utama
0,1,Load corpus,LULUS,Corpus dan hash,2
1,2,Explore corpus,LULUS,Statistik/top-20/histogram/Zipf utama,3
2,3,Preprocessing,LULUS,Contoh preprocessing utama,4
3,4,Split,LULUS,Indeks split identik,5
4,5,Boundary,LULUS,Padding model dan corpus mini,6
5,6,Unknown words,LULUS,contoh_unk.csv,7
6,7,Generate n-grams,LULUS,Contoh n-gram utama dan assert,8
7,8,Count n-grams,LULUS,Counter event train,9
8,9,MLE,LULUS,perhitungan_token.csv,10
9,10,Next-word prediction,LULUS,prediksi_perbandingan.csv,11


16 langkah: lulus; core dan 157 hasil lama tetap identik.
